# ML Learning Coach (English): Part 3, Reload the adapter and compare before/after
Loads the adapter **from disk** with `PeftModel.from_pretrained`, runs the same test prompts and compares with the baseline.
Outputs: `finetuned_outputs.json` and `before_after.md` (for the README).
Runtime > Change runtime type > **T4 GPU**.

### 1) Libraries

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate

### 2) Connect Drive and check the adapter

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/ml-coach-en"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(os.listdir(DRIVE_DIR))
ADAPTER_DIR = f"{DRIVE_DIR}/adapter"
print("Adapter folder:", os.listdir(ADAPTER_DIR))
assert "adapter_config.json" in os.listdir(ADAPTER_DIR), 'Adapter not found. Run 02_train_qlora first (and its LAST cell).'

### 3) Get the files

In [ ]:
import shutil
from google.colab import files

def fetch(name):
    if os.path.exists(name):
        return
    p = f"{DRIVE_DIR}/{name}"
    if os.path.exists(p):
        shutil.copy(p, name)
        return
    print(f"{name} not found, please upload it:")
    up = files.upload()
    k = list(up)[0]
    if k != name:
        shutil.move(k, name)
    shutil.copy(name, p)

for name in ["train.jsonl", "test_prompts.json", "baseline_outputs.json"]:
    fetch(name)
print("Files ready")

### 4) Read the data

In [ ]:
import json

SYSTEM = json.loads(open("train.jsonl", encoding="utf-8").readline())["messages"][0]["content"]
tests = json.load(open("test_prompts.json", encoding="utf-8"))
before = {b["id"]: b["answer"] for b in json.load(open("baseline_outputs.json", encoding="utf-8"))}
print("Test prompts:", len(tests), "| baseline answers:", len(before))

### 5) Load the base model, then attach OUR adapter
This is the required check: the model is rebuilt from the **adapter on disk**, independent of the training session.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print("Adapter loaded:", list(model.peft_config))
print("GPU memory (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

### 6) Run the same prompts with the fine-tuned model

In [ ]:
def generate(messages, max_new_tokens=300):
    msgs = [{"role": "system", "content": SYSTEM}] + messages
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, temperature=None, top_p=None, top_k=None)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


after = []
for t in tests:
    ans = generate(t["messages"])
    after.append({"id": t["id"], "type": t["type"], "last_user": t["messages"][-1]["content"], "answer": ans})
    print("=" * 70)
    print(f"[{t['id']}]  ({t['type']})")
    print("STUDENT:", t["messages"][-1]["content"])
    print("\n--- BEFORE (base model) ---\n" + before[t["id"]])
    print("\n--- AFTER (fine-tuned) ---\n" + ans)

### 7) Simple measurements: format, verdict correctness, memory
Not a full quality measure, but objective evidence: does the model follow the format? Does it give the right verdict on wrong claims? Does it give a NEW example when asked?

In [ ]:
MODE1 = ["Simple explanation:", "Example:", "Check yourself:"]
MODE2 = ["Verdict:", "What is right:", "What to improve:", "Think about it:"]
VERDICTS = ["Correct", "Partly correct", "Wrong"]
EXPECTED = {"check_overfitting": "Wrong", "check_knn": "Wrong", "check_regularization": "Correct",
            "check_random_forest": "Wrong", "check_split": "Correct"}


def labels_ok(a, labels):
    lines = a.split("\n")
    return all(any(l.startswith(x) for l in lines) for x in labels)


def verdict_of(a):
    first = a.split("\n")[0]
    return first.replace("Verdict:", "").strip() if first.startswith("Verdict:") else None


def format_ok(tid, a):
    if tid.startswith("off"):
        return (not a.startswith(("Simple explanation:", "Verdict:"))) and len(a.split()) <= 60 and a.count("?") == 1
    if tid.startswith("check") or tid == "follow_answer":
        return labels_ok(a, MODE2) and verdict_of(a) in VERDICTS
    return labels_ok(a, MODE1)


def example_line(a):
    for l in a.split("\n"):
        if l.startswith("Example:"):
            return l
    return None


after_d = {a["id"]: a["answer"] for a in after}
ids = [t["id"] for t in tests]

fmt_before = sum(format_ok(i, before[i]) for i in ids)
fmt_after = sum(format_ok(i, after_d[i]) for i in ids)
ver_before = sum(verdict_of(before[i]) == EXPECTED[i] for i in EXPECTED)
ver_after = sum(verdict_of(after_d[i]) == EXPECTED[i] for i in EXPECTED)

prev = next(t for t in tests if t["id"] == "follow_example")["messages"][1]["content"]
def new_example(a):
    e = example_line(a)
    return bool(e) and e != example_line(prev)

print(f"Answers in the right format:     before {fmt_before}/{len(ids)}  ->  after {fmt_after}/{len(ids)}")
print(f"Correct verdict (check mode):    before {ver_before}/{len(EXPECTED)}  ->  after {ver_after}/{len(EXPECTED)}")
print(f"Gave a NEW example (memory):     before {new_example(before['follow_example'])}  ->  after {new_example(after_d['follow_example'])}")
print()
for i in EXPECTED:
    print(f"{i:<22} expected: {EXPECTED[i]:<8} before: {verdict_of(before[i])!s:<15} after: {verdict_of(after_d[i])}")

### 8) Save the results
`before_after.md` can be pasted into the README. `finetuned_outputs.json` is evidence.

In [ ]:
with open("finetuned_outputs.json", "w", encoding="utf-8") as f:
    json.dump(after, f, ensure_ascii=False, indent=2)

lines = ["# Before / After comparison", "",
         f"- Answers in the right format: **{fmt_before}/{len(ids)} -> {fmt_after}/{len(ids)}**",
         f"- Correct verdict (check mode): **{ver_before}/{len(EXPECTED)} -> {ver_after}/{len(EXPECTED)}**", ""]
for t in tests:
    i = t["id"]
    lines += [f"## {i}  ({t['type']})", "", "**Student:**", "", "```", t["messages"][-1]["content"], "```", "",
              "**Before (base model):**", "", "```", before[i], "```", "",
              "**After (fine-tuned):**", "", "```", after_d[i], "```", ""]
open("before_after.md", "w", encoding="utf-8").write("\n".join(lines))

for name in ["finetuned_outputs.json", "before_after.md"]:
    shutil.copy(name, f"{DRIVE_DIR}/{name}")
    files.download(name)
print("Saved")